In [1]:
import ollama
import os

import faiss
import numpy as np

In [2]:
!pip install pypdf python-docx

In [3]:
import os
from pypdf import PdfReader
from docx import Document

folder_path = "legal"

documents = []

for filename in os.listdir(folder_path):

    file_path = os.path.join(folder_path, filename)

    # Load PDF
    if filename.endswith(".pdf"):

        reader = PdfReader(file_path)

        text = ""

        for page in reader.pages:
            text += page.extract_text() + "\n"

        documents.append({
            "text": text,
            "source": filename
        })

    # Load Word document
    elif filename.endswith(".docx"):

        doc = Document(file_path)

        text = ""

        for paragraph in doc.paragraphs:
            text += paragraph.text + "\n"

        documents.append({
            "text": text,
            "source": filename
        })

print("Total documents loaded:", len(documents))

for doc in documents:
    print("\nSource:", doc["source"])
    print("Text preview:", doc["text"][:200])

Total documents loaded: 8

Source: 01_Criminal_Case_File_Chowdary.docx
Text preview: Meridian & Rao Legal Associates, Hyderabad
Internal Case File — Privileged & Confidential
State of Telangana vs. Vikram Chowdary
Criminal Case No. CC/482/2024 — Cheating and Forgery
Illustrative / syn

Source: 01_Criminal_Case_File_Chowdary.pdf
Text preview: Meridian & Rao Legal Associates, Hyderabad — Internal Case File — CC/482/2024
Meridian & Rao Legal Associates, Hyderabad
Internal Case File — Privileged & Confidential
State of Telangana vs. Vikram Ch

Source: 02_Commercial_Dispute_Case_File_BlueWave.docx
Text preview: Meridian & Rao Legal Associates, Hyderabad
Internal Case File — Privileged & Confidential
BlueWave Constructions Pvt. Ltd. vs. Orion Infra Solutions Pvt. Ltd.
Commercial Suit No. CS(COMM)/112/2024 — B

Source: 02_Commercial_Dispute_Case_File_BlueWave.pdf
Text preview: Meridian & Rao Legal Associates, Hyderabad — Internal Case File — CS(COMM)/112/2024
Meridian & Rao Legal Associates, 

In [4]:
def chunk_documents(documents):

    chunk_size = 800
    chunk_overlap = 100

    chunks = []

    for document in documents:

        text = document["text"]
        source = document["source"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk = text[start:end]

            chunks.append({ "source": source, "text": chunk})

            start = end - chunk_overlap

    return chunks


In [5]:
chunks = chunk_documents(documents)

print("\nTotal chunks:",len(chunks))


Total chunks: 82


In [6]:
test_questions = [

    {
        "question": "What are the key terms and conditions mentioned in the agreement?",
        "expected_source": "contract.txt"
    },

    {
        "question": "What are the rights and responsibilities of the parties involved?",
        "expected_source": "contract.txt"
    },

    {
        "question": "What are the conditions for termination of the agreement?",
        "expected_source": "contract.txt"
    },

    {
        "question": "What are the employee rights and protections under the applicable law?",
        "expected_source": "labour_law.txt"
    },

    {
        "question": "What are the legal requirements for employment and working conditions?",
        "expected_source": "labour_law.txt"
    },

    {
        "question": "What remedies are available to consumers in case of unfair practices?",
        "expected_source": "consumer_law.txt"
    },

    {
        "question": "What are the rights of consumers under the law?",
        "expected_source": "consumer_law.txt"
    },

    {
        "question": "What is the procedure for resolving a legal dispute?",
        "expected_source": "dispute_resolution.txt"
    },

    {
        "question": "What are the rules related to breach of contract?",
        "expected_source": "contract_law.txt"
    },

    {
        "question": "What are the penalties or consequences for violating the applicable law?",
        "expected_source": "legal_penalties.txt"
    }

]

In [7]:

def create_embeddings(chunks):

    embeddings = []

    for chunk in chunks:

        response = ollama.embeddings(
            model="nomic-embed-text",
            prompt=chunk["text"]
        )

        embeddings.append(
            response["embedding"]
        )

    return np.array(
        embeddings,
        dtype="float32"
    )


In [8]:
chunk_embeddings = create_embeddings(chunks)

print( "\nEmbedding shape:",chunk_embeddings.shape)


Embedding shape: (82, 768)


In [9]:

def create_faiss_index(embeddings):

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2( dimension)
    index.add(embeddings)
    return index

In [10]:

index = create_faiss_index(chunk_embeddings)

print("Vectors stored in FAISS:",index.ntotal)

Vectors stored in FAISS: 82


In [11]:
question = "What are the key terms and conditions mentioned in the agreement?"
print("\nQuestion:", question)


Question: What are the key terms and conditions mentioned in the agreement?


In [12]:

def embed_question(question):
    response = ollama.embeddings(
        model="nomic-embed-text",
        prompt=question
    )

    question_embedding = np.array(
        [response["embedding"]],
        dtype="float32"
    )

    return question_embedding


In [13]:
question_embedding = embed_question(question)

print("\nQuestion embedding shape:",question_embedding.shape)


Question embedding shape: (1, 768)


In [14]:

def search_faiss( question_embedding, index, k=3):

    distances, indices = index.search( question_embedding, k)

    return distances[0], indices[0]


In [15]:

distances, indices = search_faiss( question_embedding, index, k=3)

print( "\nDistances:", distances)
print( "Chunk numbers:", indices)


Distances: [306.98517 313.65967 322.90765]
Chunk numbers: [24 36 33]


In [16]:

def retrieve_chunks( indices, chunks):

    retrieved_chunks = []

    for i in indices:
        retrieved_chunks.append( chunks[i] )
    return retrieved_chunks


In [17]:
retrieved_chunks = retrieve_chunks(indices, chunks)

print( "\nNumber of retrieved chunks:", len(retrieved_chunks))


Number of retrieved chunks: 3


In [18]:

for i, chunk in enumerate( retrieved_chunks):

    print( f"\n===== RETRIEVED CHUNK {i + 1} =====" )

    print( "Source:", chunk["source"])

    print( "Distance:", distances[i])

    print("\nText:")

    print(chunk["text"] )


===== RETRIEVED CHUNK 1 =====
Source: 02_Commercial_Dispute_Case_File_BlueWave.docx
Distance: 306.98517

Text:
ttlement posture in light of expert report findings once expert testimony concludes
8. Internal Notes (Privileged & Confidential)
Client has indicated the commercial relationship with the defendant is unlikely to continue regardless of outcome; full recovery of damages remains the priority over any settlement discussion. Internal cost-benefit note on expert witness fees to be circulated separately.
9. Relevant Contractual and Legal Provisions
Clause 14.2 of the EPC Contract — Liquidated damages at 0.5% of contract value per week of delay, capped at 10% of contract value
Clause 18 of the EPC Contract — Force majeure, including notice requirement within 7 days of the triggering event
Section 73, Indian Contract Act, 1872 — Compensation for loss or damage caused by breach of contract
Section 

===== RETRIEVED CHUNK 2 =====
Source: 02_Commercial_Dispute_Case_File_BlueWave.pdf
Dis

In [19]:

def create_context( retrieved_chunks ):

    context = ""

    for chunk in retrieved_chunks:

        context += f"""
Source: {chunk["source"]}

{chunk["text"]}

"""

    return context


In [20]:

context = create_context( retrieved_chunks)

print("\n===== CONTEXT =====")
print(context)



===== CONTEXT =====

Source: 02_Commercial_Dispute_Case_File_BlueWave.docx

ttlement posture in light of expert report findings once expert testimony concludes
8. Internal Notes (Privileged & Confidential)
Client has indicated the commercial relationship with the defendant is unlikely to continue regardless of outcome; full recovery of damages remains the priority over any settlement discussion. Internal cost-benefit note on expert witness fees to be circulated separately.
9. Relevant Contractual and Legal Provisions
Clause 14.2 of the EPC Contract — Liquidated damages at 0.5% of contract value per week of delay, capped at 10% of contract value
Clause 18 of the EPC Contract — Force majeure, including notice requirement within 7 days of the triggering event
Section 73, Indian Contract Act, 1872 — Compensation for loss or damage caused by breach of contract
Section 


Source: 02_Commercial_Dispute_Case_File_BlueWave.pdf

mpletion cost evidence
• Evaluate settlement posture in light of e

In [21]:

def create_prompt( question, context ):

    prompt = f"""
You are an legal document question-answering assistant.

Answer the question using ONLY the information
provided in the context.

Do not use outside knowledge.

If the answer is not available in the context,
say:

"I could not find the answer in the provided documents."

Give a clear and concise answer.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt


In [22]:

prompt = create_prompt( question, context)

In [23]:

def generate_answer(prompt):

    response = ollama.chat(
 
        model="llama3.2",

        messages=[
            { "role": "user", "content": prompt }
        ],

        options={ "temperature": 0}
    )

    return response["message"]["content"]


In [24]:

answer = generate_answer( prompt)

In [25]:

print( "FINAL RAG ANSWER")

print(answer)


FINAL RAG ANSWER
The key terms and conditions mentioned in the agreement are:

1. Liquidated damages clause (Clause 14.2 of the EPC Contract): 0.5% of contract value per week of delay, capped at 10% of contract value.
2. Force majeure clause (Clause 18 of the EPC Contract): Notice requirement within 7 days of the triggering event.
3. Compensation for loss or damage caused by breach of contract (Section 73, Indian Contract Act, 1872): Compensation for loss or damage caused by breach of contract.


In [26]:

print( "\nSources used:")

for chunk in retrieved_chunks:

    print( "-", chunk["source"] )


Sources used:
- 02_Commercial_Dispute_Case_File_BlueWave.docx
- 02_Commercial_Dispute_Case_File_BlueWave.pdf
- 02_Commercial_Dispute_Case_File_BlueWave.pdf


In [27]:
expected_source="02_Commercial_Dispute_Case_File_BlueWave.pdf"


In [28]:
print(type(chunks))
print(chunks[0])

<class 'list'>
{'source': '01_Criminal_Case_File_Chowdary.docx', 'text': 'Meridian & Rao Legal Associates, Hyderabad\nInternal Case File — Privileged & Confidential\nState of Telangana vs. Vikram Chowdary\nCriminal Case No. CC/482/2024 — Cheating and Forgery\nIllustrative / synthetic case file created for training and demonstration purposes (e.g., testing retrieval-augmented generation systems). All names, dates, case numbers, and facts are fictional and do not correspond to any real person, entity, or proceeding.\n1. Case Overview\n2. Brief Facts\nThe complainant, Sunrise Textiles Pvt. Ltd., placed a purchase order dated 14 February 2024 with Chowdary Trading Co. for the supply of 12,000 meters of dyed cotton fabric, valued at ₹18,40,000, with delivery scheduled within 30 days. An advance of ₹9,20,000 (50% of the order value) was paid by RTGS on 16 February 2024 again'}


In [29]:
distances, indices = retrieved_chunks

sources = []

for i in indices:
    sources.append(chunks[i]["source"])

if expected_source in sources:
    hit_rate = 1
else:
    hit_rate = 0

print("Retrieved Sources:", sources)
print("Expected Source:", expected_source)
print("Hit Rate:", hit_rate)

ValueError: too many values to unpack (expected 2)

In [ ]:
faithfulness_prompt = f"""
Is the following answer completely supported by the context?

Context:
{context}

Answer:
{answer}

Return only 1 or 0.

1 = Supported
0 = Not supported
"""

In [ ]:
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": faithfulness_prompt
        }
    ],
    options={"temperature": 0}
)

In [ ]:
faithfulness = response["message"]["content"].strip()

print("Faithfulness:", faithfulness)

Faithfulness: 0

The answer is not supported by the context. The context mentions that the client has indicated the commercial relationship with the defendant is unlikely to continue regardless of outcome, and full recovery of damages remains the priority over any settlement discussion. This suggests that the client is prioritizing full recovery of damages over any potential settlement, which contradicts the statement that compensation for loss or damage caused by breach of contract (Section 73, Indian Contract Act, 1872) is the priority.


In [ ]:
relevance_prompt = f"""
Does the answer directly answer the question?

Question:
{question}

Answer:
{answer}

Return only 1 or 0.

1 = Yes
0 = No
"""

In [ ]:
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": relevance_prompt
        }
    ],
    options={"temperature": 0}
)

In [ ]:
answer_relevance = response["message"]["content"].strip()

print("Answer Relevance:", answer_relevance)

Answer Relevance: 1


In [ ]:
# PRECISION@3

distances, indices = retrieved_chunks

relevant = 0

for i in indices:
    if chunks[i]["source"] == expected_source:
        relevant += 1

precision_at_3 = relevant / len(indices)

print("Retrieved Sources:", [chunks[i]["source"] for i in indices])
print("Expected Source:", expected_source)
print("Precision@3:", round(precision_at_3, 2))

Retrieved Sources: ['02_Commercial_Dispute_Case_File_BlueWave.docx', '02_Commercial_Dispute_Case_File_BlueWave.pdf', '02_Commercial_Dispute_Case_File_BlueWave.pdf']
Expected Source: 02_Commercial_Dispute_Case_File_BlueWave.pdf
Precision@3: 0.67


In [ ]:
# RECALL@3

if expected_source in sources:
    recall_at_3 = 1
else:
    recall_at_3 = 0

print("Recall@3         :", recall_at_3)

Recall@3         : 1
